# Homework 2: Step-by-Step Solution (CS Edition)

This notebook has been rewritten to translate bioinformatics concepts into computer science analogies.
If you know what strings, arrays, diffs, log files, and compression are, but not what 'exome', 'neoplasm', or 'germline' are, this is for you.

Expected data location in WSL:
```bash
~/compgenomicsws_hw2
```
If your files are somewhere else, update `DATA_DIR` in the first code cell.

## 0. Big picture for a computer science student

This assignment is basically a file-format and data-inspection exercise. You're analyzing gigantic text files and verifying basic properties.

The story is:
1. We have two datasets from one person: one from a tumor, and one from nearby normal tissue.
   - **Analogy**: You have a modified, buggy program (the tumor) and the original program (normal tissue). You want to `diff` them to find what changed.
2. The sequencing machine produced compressed **FASTQ files**. A FASTQ file is just a giant log file where every entry (a *read*) takes exactly four lines. Each read is a short substring of the DNA.
3. The reads are **paired-end**, meaning each DNA substring was read from both the front and the back. Imagine taking a string, getting a chunk of size 500 characters, and using one scanner to read the first 100 characters left-to-right (`R1`) and another scanner to read the last 100 characters right-to-left (`R2`).
4. **FastQC** is like a linter or test suite. It checks whether the raw data has systemic errors (like hardware artifacts during reading).
5. `hg38` is the human reference genome. Think of it as the canonical standard library/repository containing the 'default' string of human DNA. We inspect it as a **FASTA file**, which is essentially a text-based hash map (keys are >headers, values are the actual DNA strings).

Vocabulary translation:
- **Sample**: One target payload that was read. (e.g., the tumor biopsy).
- **Read**: One short string of DNA (usually 100-150 chars). A FASTQ file is just millions of these reads.
- **Base**: One character in the string. The alphabet is `{A, C, G, T}`.
- **bp (base pair)**: The length of the string. A 100 bp read is a string of length 100.
- **Exome/WXS**: The exome refers to the roughly 1.5% of the genome that actually contains functioning, encoding 'code' (genes that make proteins). The rest of the genome is mostly 'comments' or evolutionary baggage. **WXS (Whole Exome Sequencing)** means we only sequence this 1.5%. WGS (Whole Genome Sequencing) means we read everything.


## 1. Setup and Files From the README

The README tells us to use:

- `Metadata_18.csv` for sample metadata.
- Four paired-end FASTQ files: normal `R1/R2` and tumor `R1/R2`.
- FastQC for raw-read quality control.
- `hg38.fa.gz` as the human reference genome.

We will use Python for reproducible parsing and shell commands only where they mirror the original assignment workflow. The FASTQ and FASTA files are gzip-compressed, so Python's `gzip` module can read them directly without manually unzipping huge files.

What this setup cell does:

- Points Python at the assignment folder and downloaded data folder.
- Gives friendly names to the four FASTQ files.
- Checks whether the expected files exist before we spend time parsing them.

In [1]:
from pathlib import Path
import os
import gzip
import csv
import zipfile
import subprocess
import statistics
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")
import pandas as pd
import plotly.express as px
from IPython.display import display

DATA_DIR = Path.home() / "compgenomicsws_hw2"

# VS Code/Jupyter may start either in this assignment folder or at the repo root.
# This small check makes the notebook work in both cases.
ASSIGNMENT_DIR = Path.cwd()
if not (ASSIGNMENT_DIR / "Metadata_18.csv").exists():
    candidate = ASSIGNMENT_DIR / "Lesson3[IntroNGS]+Assignment2"
    if candidate.exists():
        ASSIGNMENT_DIR = candidate

METADATA = ASSIGNMENT_DIR / "Metadata_18.csv"

FASTQ_FILES = {
    "normal_R1": DATA_DIR / "ERR4833597_1.fastq.gz",
    "normal_R2": DATA_DIR / "ERR4833597_2.fastq.gz",
    "tumor_R1": DATA_DIR / "ERR4833621_1.fastq.gz",
    "tumor_R2": DATA_DIR / "ERR4833621_2.fastq.gz",
}

HG38 = DATA_DIR / "hg38.fa.gz"
FASTQC_DIR = DATA_DIR / "fastqc"

print(f"Assignment directory: {ASSIGNMENT_DIR}")
print(f"Data directory:       {DATA_DIR}")
print("\nFASTQ files:")
for label, path in FASTQ_FILES.items():
    print(f"{label:10s} {path} exists={path.exists()}")
print(f"\nhg38 exists={HG38.exists()} path={HG38}")



Assignment directory: /mnt/c/Users/orgrd/workspace/repos/CompGenomicsWS/Lesson3[IntroNGS]+Assignment2
Data directory:       /home/orgr/compgenomicsws_hw2

FASTQ files:
normal_R1  /home/orgr/compgenomicsws_hw2/ERR4833597_1.fastq.gz exists=True
normal_R2  /home/orgr/compgenomicsws_hw2/ERR4833597_2.fastq.gz exists=True
tumor_R1   /home/orgr/compgenomicsws_hw2/ERR4833621_1.fastq.gz exists=True
tumor_R2   /home/orgr/compgenomicsws_hw2/ERR4833621_2.fastq.gz exists=True

hg38 exists=True path=/home/orgr/compgenomicsws_hw2/hg38.fa.gz


## 2. Metadata Questions

We start with `Metadata_18.csv`. This file carries the attributes/schema context for our FASTQ log files.

### Question 1
> The file `Metadata_18.csv` contains additional meta information about the files we are downloading. What does `neoplasm` and `normal tissue adjacent to neoplasm` mean?

**CS Perspective**: 
- **Neoplasm**: This means the tumor. Think of it as the 'modified, buggy' version of the DNA.
- **Normal tissue adjacent to neoplasm**: This is the 'baseline' or 'control' version of the DNA, taken from healthy tissue nearby.

**Why compare them?**
Every human has roughly 3-4 million normal differences in their DNA compared to the `hg38` reference. If you only sequence the tumor and diff it against the reference genome, you will get 4 million 'bugs', but 99.9% of them are just the user's normal baseline variations (like their personalized config file).
By diffing the tumor (Neoplasm) against the patient's own healthy tissue (Normal), you filter out the baseline inherited variations (**germline** mutations) and are left ONLY with the specific bugs that caused the cancer (**somatic** mutations).

Important vocabulary:
- **Germline genotype**: The default software shipped from the factory (inherited DNA).
- **Somatic genotype**: Hotfixes applied at runtime (mutations acquired during life that may cause cancer).


In [2]:
# The CSV contains duplicate column names, so csv.DictReader is awkward here.
# We read rows manually and display only the columns that answer the homework questions.
with METADATA.open(newline="") as fh:
    reader = csv.reader(fh)
    header = next(reader)
    rows = list(reader)

interesting_columns = [
    "Characteristics[individual]",
    "Characteristics[organism part]",
    "Characteristics[sampling site]",
    "Characteristics[disease]",
    "Characteristics[genotype]",
    "Comment[LIBRARY_LAYOUT]",
    "Comment[LIBRARY_STRATEGY]",
    "Scan Name",
    "Comment[FASTQ_URI]",
]

indices = [header.index(col) for col in interesting_columns]

for row in rows:
    print("-" * 80)
    for col, idx in zip(interesting_columns, indices):
        print(f"{col:35s}: {row[idx]}")

--------------------------------------------------------------------------------
Characteristics[individual]        : AD0699_18
Characteristics[organism part]     : uterine cervix
Characteristics[sampling site]     : normal tissue adjacent to neoplasm
Characteristics[disease]           : cervical cancer
Characteristics[genotype]          : germline genotype
Comment[LIBRARY_LAYOUT]            : PAIRED
Comment[LIBRARY_STRATEGY]          : WXS
Scan Name                          : AD0699_18N_R1.fastq.gz
Comment[FASTQ_URI]                 : ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR483/007/ERR4833597/ERR4833597_1.fastq.gz
--------------------------------------------------------------------------------
Characteristics[individual]        : AD0699_18
Characteristics[organism part]     : uterine cervix
Characteristics[sampling site]     : normal tissue adjacent to neoplasm
Characteristics[disease]           : cervical cancer
Characteristics[genotype]          : germline genotype
Comment[LIBRARY_LAY

### Data exploration: what is in the metadata?

Before answering, it helps to ask: "What are the rows, and which columns tell me what each FASTQ file represents?"

This exploration cell turns the important metadata columns into a compact table-like view. It also counts sample types and library strategies. The goal is to make the biological labels feel like normal categorical data.

In [3]:
# Build small dictionaries for the columns we care about.
metadata_records = []
for row in rows:
    metadata_records.append({col: row[idx] for col, idx in zip(interesting_columns, indices)})

for i, record in enumerate(metadata_records, start=1):
    print(f"Sample {i}")
    print(f"  run/accession: {record['Scan Name']}")
    print(f"  tissue label:  {record['Characteristics[organism part]']}")
    print(f"  genotype:      {record['Characteristics[genotype]']}")
    print(f"  layout:        {record['Comment[LIBRARY_LAYOUT]']}")
    print(f"  strategy:      {record['Comment[LIBRARY_STRATEGY]']}")
    print()

# Simple categorical counts. This is the metadata as ordinary tabular data.
def count_values(records, key):
    counts = {}
    for record in records:
        counts[record[key]] = counts.get(record[key], 0) + 1
    return counts

summary_rows = []
for label, key in [
    ("Tissue labels", "Characteristics[organism part]"),
    ("Genotypes", "Characteristics[genotype]"),
    ("Library strategies", "Comment[LIBRARY_STRATEGY]"),
    ("Library layouts", "Comment[LIBRARY_LAYOUT]"),
]:
    counts = count_values(metadata_records, key)
    print(label)
    for value, count in counts.items():
        print(f"  {value}: {count}")
        summary_rows.append({"category": label, "value": value, "count": count})
    print()

metadata_summary_df = pd.DataFrame(summary_rows)
fig = px.bar(
    metadata_summary_df,
    x="value",
    y="count",
    color="category",
    facet_col="category",
    facet_col_wrap=2,
    title="Metadata categories used to understand the samples",
    labels={"value": "metadata value", "count": "number of rows"},
)
fig.update_xaxes(matches=None, tickangle=25)
fig.update_yaxes(dtick=1)
fig.show()

Sample 1
  run/accession: AD0699_18N_R1.fastq.gz
  tissue label:  uterine cervix
  genotype:      germline genotype
  layout:        PAIRED
  strategy:      WXS

Sample 2
  run/accession: AD0699_18N_R2.fastq.gz
  tissue label:  uterine cervix
  genotype:      germline genotype
  layout:        PAIRED
  strategy:      WXS

Sample 3
  run/accession: AD0729_18T_R1.fastq.gz
  tissue label:  uterine cervix
  genotype:      somatic genotype
  layout:        PAIRED
  strategy:      WXS

Sample 4
  run/accession: AD0729_18T_R2.fastq.gz
  tissue label:  uterine cervix
  genotype:      somatic genotype
  layout:        PAIRED
  strategy:      WXS

Tissue labels
  uterine cervix: 4

Genotypes
  germline genotype: 2
  somatic genotype: 2

Library strategies
  WXS: 4

Library layouts
  PAIRED: 4



### Answer 1

In this metadata, `neoplasm` means the sample taken from the tumor tissue. A neoplasm is an abnormal tissue growth; for this assignment, you can read it as "the tumor sample".

`normal tissue adjacent to neoplasm` means tissue taken from near the tumor, from the same patient, that is not labeled as tumor. It is used as the patient's own baseline/control sample.

This matters because every person has many normal DNA differences. By comparing tumor vs same-patient normal, we can later ask: "which DNA differences are specific to the tumor?"

Simple mapping:

- `ERR4833621` / `AD0729_18T` = tumor sample
- `ERR4833597` / `AD0699_18N` = matched normal/control sample

### Question 2
> The 23rd column (`LIBRARY_STRATEGY`) is marked as `WXS`. What is Whole Exome Sequencing (`WXS`/`WES`)? What would be the FASTQ file sizes if we did Whole Genome Sequencing (`WGS`) instead?


### Answer 2

**WXS/WES (Whole Exome Sequencing)** means we are using a filter to only sequence the 'Exome'. The exome is the small fraction (approx. 1-2%) of the human genome that actually codes for proteins. In CS terms, we are only dumping the executable code segments and ignoring the massive blobs of dead code, padding, and comments in the binary.

If we did **WGS (Whole Genome Sequencing)**, we would sequence the entire 3.2 billion character string. Because we need redundancy (coverage) to accurately read the DNA, a single compressed FASTQ file for WGS is usually around 30GB to 100GB, compared to the ~1-3GB sizes you see for WXS.


## 3. FASTQ Headers

### Question 3
> Once the downloads have completed start by simply looking at each file and making sure the data format is OK. Make sure the read titles make sense and match between `R1` and `R2`.
>
> What is the difference between the header of the first read in `R1` and the header of the first read in `R2`?

**The FASTQ Format**:
A FASTQ file is purely a text-based format for storing reads and their confidence scores. Every single read record spans **exactly 4 lines**:
1. `@<Identifier and header info>`
2. `The actual string of characters (A, C, G, T)`
3. `+ (sometimes repeating the header, mostly empty)`
4. `An ASCII-encoded string of confidence probabilities (Phred score) matching the sequence string length 1-to-1`

**Paired End Reads**: Because we read the front and back of the string, you get two FASTQ files. `R1` holds the front reads, `R2` holds the back reads. The line numbers line up perfectly (Read 100 in file R1 is paired with Read 100 in file R2).


In [4]:
def first_fastq_record(path):
    with gzip.open(path, "rt") as fh:
        return [next(fh).rstrip() for _ in range(4)]

for label in ["normal_R1", "normal_R2", "tumor_R1", "tumor_R2"]:
    record = first_fastq_record(FASTQ_FILES[label])
    print("=" * 80)
    print(label)
    print("\n".join(record))

normal_R1
@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/1
GGCGGCGGCGGCGGCGGCGGCGGCGGCGGCCGGGGCTGGTGGGGGAGGGGGGGTGGCCCCTGATCCTCCGGTAGCGAAGGGCTAGGGGCACACGCCCAGC
+
BBBBFBF<FF7B########################################################################################
normal_R2
@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/2
GCCCTCCCTGCTCCCGGGCGGCTCTGGAGGAGCTCGGTGTTTGCCTCTAGCCCTTCGCTACCGGAGGATCAGGTTCCTCCCCTCCTCCCCCTCCAGCTCC
+
BBBFFBF<FBFFFFIF7FIFIF7FBB<0<<FBBBFBBB<'7<B<B<<B<770<BBB7<BBBBB<0000B<BBB0<<0<0707<7''007'777B<'0<BB
tumor_R1
@ERR4833621.1 HISEQ:451:C78DHACXX:1:1101:10000:16167/1
CCCACAGCAGCCCAGTGAAGAGTCCTGTTATCTTTTCGGTTTGACAGATGAGGAGATGGAGGCCAGAGAGGATGGGCCACCGGCCAAGGTCATTGTCCTG
+
BBBFFFFFFFFFFIIFIFIIIIFFIFIFFFIFFIIIIIIFIIIIIIIIIIIFIIIFFIII<FFFFFFFBFFFFFFB<BBBFBFBFFF<B0<BBFFFFFBB
tumor_R2
@ERR4833621.1 HISEQ:451:C78DHACXX:1:1101:10000:16167/2
GGGCAGTGACTGAATCCCCAAGAAAGTAGAGCCATCAAGGTCAGGGCTCTGTGGGCACAGGGTTGGGATTTAAATCCCAGCTCCACAGGACAATGACCTT
+
BBBFFFFFFFFFFIIIIIIFIIFIIIB

### Data exploration: decode a FASTQ record

The quality string is not random decoration. It encodes one quality score per DNA base. A higher score means the sequencer is more confident about that base call.

This cell prints the first record as aligned positions so you can see that sequence length and quality-string length match.

In [5]:
### Answer 3

For the normal sample, the first `R1` and `R2` headers are:

- `@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/1`
- `@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/2`

The read identifier string is strictly identical. The only difference is the suffix `/1` (which denotes the 'forward' read) and `/2` (which denotes the 'reverse' read). This is how tools know they belong to the same fragment of DNA in memory.


SyntaxError: invalid syntax (879505.py, line 3)

### Answer 3

For the normal sample, the first `R1` and `R2` headers are:

- `@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/1`
- `@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/2`

The read identifier is the same. The suffix differs: `/1` marks `R1` and `/2` marks `R2`. That tells us these are the two mates from the same original DNA fragment.

## 4. Read Length and Read Counts

### Question 4

> What is the read length? How many reads are there per file?

The read length is the length of the sequence line in a FASTQ record. If the sequence line has 100 DNA letters, the read length is `100 bp`.

The read count is the total number of FASTQ lines divided by `4`, because each read occupies exactly four lines.

A paired-end sample has two FASTQ files. If `R1` and `R2` have the same number of reads, that means every read in one file has its mate in the other file. So the number of read pairs is the same as the read count in either mate file, not the sum of both files.

In [ ]:
def fastq_stats(path):
    line_count = 0
    first_read_length = None
    with gzip.open(path, "rt") as fh:
        for line_count, line in enumerate(fh, start=1):
            if line_count == 2:
                first_read_length = len(line.rstrip())
    return {
        "read_length": first_read_length,
        "line_count": line_count,
        "read_count": line_count // 4,
    }

stats = {label: fastq_stats(path) for label, path in FASTQ_FILES.items()}

for label, result in stats.items():
    print(f"{label:10s} read_length={result['read_length']} read_count={result['read_count']:,}")

### Data exploration: compare read counts visually

The assignment asks for read length and read count. A quick plot helps you notice two important things:

- `R1` and `R2` have matching counts inside each sample.
- The tumor sample has more reads than the normal sample.

In [ ]:
read_stats_rows = []
for label, result in stats.items():
    sample = "normal" if label.startswith("normal") else "tumor"
    mate = "R1" if label.endswith("R1") else "R2"
    read_stats_rows.append({
        "file_label": label,
        "sample": sample,
        "mate": mate,
        "read_count": result["read_count"],
        "read_length": result["read_length"],
    })

read_stats_df = pd.DataFrame(read_stats_rows)
display(read_stats_df)

fig = px.bar(
    read_stats_df,
    x="file_label",
    y="read_count",
    color="sample",
    pattern_shape="mate",
    text="read_count",
    title="Read counts per FASTQ file",
    labels={"file_label": "FASTQ file", "read_count": "reads"},
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(yaxis_tickformat=",", xaxis_tickangle=25)
fig.show()

fig = px.bar(
    read_stats_df,
    x="file_label",
    y="read_length",
    color="sample",
    pattern_shape="mate",
    text="read_length",
    title="Read length per FASTQ file",
    labels={"file_label": "FASTQ file", "read_length": "bp"},
)
fig.update_traces(texttemplate="%{text} bp", textposition="outside")
fig.update_layout(yaxis_range=[0, 125], xaxis_tickangle=25)
fig.show()

### Data exploration: sample GC content and average quality

Bioinformatics files are often too large to fully inspect by eye. A common approach is to sample the first few thousand reads and compute simple summaries.

Here we estimate two intuitive properties:

- **GC content**: fraction of bases that are `G` or `C`.
- **Mean Phred quality**: average confidence score over a read.

In [ ]:
### Answer 4

All four files have `100 bp` reads. (They are strings of length 100).

Read counts (Number of rows / 4):

- `ERR4833597_1.fastq.gz` (Normal Front): `28,927,775`
- `ERR4833597_2.fastq.gz` (Normal Back): `28,927,775`
- `ERR4833621_1.fastq.gz` (Tumor Front): `36,383,853`
- `ERR4833621_2.fastq.gz` (Tumor Back): `36,383,853`

Notice that the read counts between `_1` and `_2` of the same sample are exactly identical. This perfectly validates the paired-end property.


### Answer 4

All four files have `100 bp` reads.

Read counts:

- `ERR4833597_1.fastq.gz`: `28,927,775`
- `ERR4833597_2.fastq.gz`: `28,927,775`
- `ERR4833621_1.fastq.gz`: `36,383,853`
- `ERR4833621_2.fastq.gz`: `36,383,853`

Because this is paired-end data, the normal sample has `28,927,775` read pairs and the tumor sample has `36,383,853` read pairs.

## 5. Run FastQC Before Quality Questions

The README asks us to run FastQC before answering the next two questions.

> Run FastQC on all the files. Did you get the number of reads per file correct?

FastQC is a quality-control tool for raw sequencing files. It does not solve the biological question; it tells us whether the input data looks technically reasonable.

FastQC produces HTML reports and `.zip` archives. The zip archives contain `summary.txt` and `fastqc_data.txt`, which are easy to parse programmatically.

If FastQC has already been run, this cell can be skipped. It is safe to run again; it will overwrite the reports.

In [ ]:
## 6. FastQC Quality Interpretation

The summary tells us whether each QC module passed, warned, or failed.

### Question 5
> Check the `Per base sequence quality`. Do the overall base qualities look ok? Do you notice any strange behavior?

### Question 6
> Does the `Per base sequence content` behave as expected?

**CS Perspective**: 
- **Per base sequence quality**: Imagine you have an optical scanner (the sequencer). Sometimes the scanner's lens gets smudged, or its laser gets weak towards the end of a long scan. The 'Phred Quality Score' is the hardware's internal probability that it made an error on that specific character. We want to check if the error rate strictly spikes at a particular character index (e.g., at index 50).
- **Per base sequence content**: If you look at an array of all reads at index `i`, what percentage of the characters are A, C, G, or T? Usually, DNA is fairly balanced. If index `0` through `10` are 90% 'A', there is a systematic padding/adapter sequence skewing the data.


## 6. FastQC Quality Interpretation

The summary tells us whether each QC module passed, warned, or failed.

### Question 5

> Check the `Per base sequence quality`. Do the overall base qualities look ok? Do you notice any strange behavior?

### Question 6

> Does the `Per base sequence content` behave as expected?

Plain-English meanings:

- **Per base sequence quality**: are the letters reliable at each position along the read?
- **Per base sequence content**: at each position, do the proportions of `A`, `C`, `G`, and `T` look reasonable?

A warning is not automatically a disaster. Real sequencing data often has small warnings, especially near the start or end of reads.

In [ ]:
def read_fastqc_summary(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        summary_name = next(name for name in zf.namelist() if name.endswith("summary.txt"))
        text = zf.read(summary_name).decode()
    return [line.split("\t") for line in text.strip().splitlines()]

for zip_path in sorted(FASTQC_DIR.glob("*_fastqc.zip")):
    print("=" * 80)
    print(zip_path.name)
    for status, module, filename in read_fastqc_summary(zip_path):
        if module in {"Basic Statistics", "Per base sequence quality", "Per base sequence content", "Per tile sequence quality", "Per sequence GC content"}:
            print(f"{status:4s} {module}")

In [ ]:
def extract_fastqc_module(zip_path, module_name):
    with zipfile.ZipFile(zip_path) as zf:
        data_name = next(name for name in zf.namelist() if name.endswith("fastqc_data.txt"))
        lines = zf.read(data_name).decode().splitlines()

    capture = False
    module_lines = []
    for line in lines:
        if line.startswith(f">>{module_name}"):
            capture = True
        if capture:
            module_lines.append(line)
        if capture and line == ">>END_MODULE":
            break
    return module_lines

for zip_path in sorted(FASTQC_DIR.glob("*_fastqc.zip")):
    print("=" * 80)
    print(zip_path.name)
    for module in ["Basic Statistics", "Per base sequence quality", "Per base sequence content"]:
        lines = extract_fastqc_module(zip_path, module)
        print(lines[0])
    print()

### Data exploration: plot FastQC quality and base composition

The HTML FastQC reports contain plots, but we can also parse the underlying text. This makes the assignment more transparent: the plot is just numbers in `fastqc_data.txt`.

The next cell plots:

- mean quality by read position
- base composition by read position for one representative file

In [ ]:
### Answer 5

Overall base quality is solid. All four files pass `Per base sequence quality`.

There's a well-known technical phenomenon where quality degrades linearly towards the 3' end (the end of the string). This is because chemical reagents lose their potency over time. However, there are visible 'dips' in quality around indices (cycles) 50-53 in R2, and 76-77 in R1. These are likely mechanical or fluidic anomalies during the sequencing run.


### Answer 6

The beginning of the reads shows a highly skewed composition for the first ~10-15 cycles, after which the proportions stabilize (the lines flatten out parallel to each other).

This behavior is actually completely 'expected' for many RNA/Exome sequencing protocols. They use pseudo-random 'primers' (like a magic byte sequence to kick off reading) that aren't perfectly random, creating a statistical bias at the start of the string. So it passes FastQC, even if it looks odd at first glance.


## 7. Reference Genome Questions

The README next asks us to inspect the downloaded human reference genome FASTA file, `hg38.fa.gz`.

### Question 7
> What is `hg38`?

### Question 8
> How many sequences are present in the human reference genome you just downloaded?

### Question 9
> Just by looking at the sequence names can you understand what sequences they are?

### Question 10
> How many nucleotides are there in chromosome 1? How many in chromosome 20?

**CS Perspective: FASTA vs FASTQ**
A **FASTA** file is basically a JSON dictionary dumped in text format. A line starting with `>` is the key (the name of the sequence). Every line that follows until the next `>` is the actual sequence string (the value). Notice there's no `+` line and no quality score string like in FASTQ, because the reference genome isn't a 'scan' of a patient—it's the canonical, idealized standard.

We treat `hg38` (also known as `GRCh38`) as the global coordinate map. For example, if we say 'Mutation at chr1, position 1,000,432', it means look up the key `>chr1`, scan to the millionth character, and see what changed.


## 7. Reference Genome Questions

The README next asks us to inspect the downloaded human reference genome FASTA file, `hg38.fa.gz`.

### Question 7

> What is `hg38`?

### Question 8

> How many sequences are present in the human reference genome you just downloaded?

### Question 9

> Just by looking at the sequence names can you understand what sequences they are?

### Question 10

> How many nucleotides are there in chromosome 1? How many in chromosome 20?

`hg38` is the UCSC name for the `GRCh38` human reference genome assembly. In practical terms, this is the version of the human genome we use as a coordinate system.

FASTA files store each sequence with a header beginning with `>`, followed by nucleotide sequence lines. Counting headers gives the number of sequences in the reference.

Important: the reference genome is not stored as one giant string. It is stored as many named sequences: main chromosomes like `chr1`, sex chromosomes like `chrX`, and extra scaffolds/contigs that represent pieces that are harder to place cleanly.

In [ ]:
def fasta_lengths(path):
    lengths = {}
    current_name = None
    current_length = 0

    with gzip.open(path, "rt") as fh:
        for line in fh:
            if line.startswith(">"):
                if current_name is not None:
                    lengths[current_name] = current_length
                current_name = line[1:].split()[0]
                current_length = 0
            else:
                current_length += len(line.strip())

    if current_name is not None:
        lengths[current_name] = current_length

    return lengths

lengths = fasta_lengths(HG38)
print(f"Number of sequences: {len(lengths)}")
print(f"chr1 length:  {lengths['chr1']:,}")
print(f"chr20 length: {lengths['chr20']:,}")

In [6]:
names = list(lengths)
print("First 25 sequence names:")
for name in names[:25]:
    print(name)

print("\nLast 25 sequence names:")
for name in names[-25:]:
    print(name)

NameError: name 'lengths' is not defined

### Data exploration: chromosome and contig lengths

The reference contains 455 named sequences, but they are not all equally important or equally large. Plotting lengths makes the structure obvious: the main chromosomes dominate, while many auxiliary contigs are much smaller.

In [ ]:
### Answer 7

`hg38` is UCSC's alias for the human reference genome assembly `GRCh38`. It is essentially the canonical 'v38' string representation of the human DNA. Any new person sequenced is diffed against this exact version to map where their reads go.


### Answer 7

`hg38` is UCSC's name for the human reference genome assembly `GRCh38`. It is a standard reference coordinate system for human DNA.

### Answer 9

Yes, the keys give it away:
- keys `chr1` through `chr22`, `chrX`, and `chrY` are the main continuous chromosomes (the major code blobs).
- keys with `_random` are strings of DNA that we know belong to a specific chromosome, but we don't know exactly what offset they should be injected into.
- keys starting with `chrUn_` ('unplaced') are loose fragments of DNA that we know represent human DNA, but we have absolutely no idea what chromosome they belong to. They are basically floating substrings.


### Answer 9

The sequence names are understandable at a high level:

- `chr1`-`chr22`, `chrX`, and `chrY` are the main chromosomes.
- Names with `_random` are unlocalized/random scaffolds, meaning we know roughly which chromosome they belong to but not their exact position.
- Names starting with `chrUn_` are unplaced contigs, meaning they are known human DNA sequence but not assigned to a chromosome position.
- Names containing `KI...` or `GL...` are accession-based auxiliary or alternate contigs.

## 8. Summary for a CS Student

Approaching bioinformatics is exactly like approaching a large-scale data processing pipeline:
1. **Define your formats**: Know the difference between **FASTQ** (your raw sensor logs) and **FASTA** (your reference lookup table).
2. **Map attributes**: The metadata file simply maps raw data blobs to variables in your experiment schema (baseline/control vs sample/payload).
3. **Run your tests**: FastQC is your automated testing framework to make sure the sensor logs aren't corrupted by hardware faults before you pass them into expensive mapping algorithms.
4. **Abstract away the biology**: You don't need to be a doctor to know that finding a somatic mutation is just string-matching the tumor log against the control log, filtering out all exact matches, and looking for single-character diffs (SNPs) against the reference map.


## 8. How I Approached the Task

When a genomics assignment feels unfamiliar, I treat it like a data-engineering problem first:

1. Find the assignment files and identify the inputs.
2. Read the metadata to map sample names to biological meaning.
3. Inspect one tiny piece of each large file before trying to process everything.
4. Use file-format rules to compute simple facts: four FASTQ lines per read, one FASTA header per sequence.
5. Run a standard QC tool, then parse its output rather than relying only on screenshots.
6. Convert final observations into short biological interpretations.

That approach keeps the biology manageable: first understand the data objects, then attach the biological names to them.